# FT 3B Solo — CommonsenseQA Direct Answer Fine-Tuning

**Purpose:** Fine-tune Qwen2.5-3B-Instruct to answer CommonsenseQA 5-choice MCQ questions **directly** (no guide, no pipeline). Evaluate on N=900 questions at seed=42 — the same split used in the paper.

| Condition | Compute | Paper result |
|---|---|---|
| Baseline (1.5B×5) | 7.5B pp | 70.7% |
| CoT (1.5B×5) | 7.5B pp | 67.7% (CoT hurts) |
| **FT 3B Solo (this run)** | **3.0B pp** | **TBD** |
| Guided pipeline | 10.5B pp | 75.8% |

CSQA has 5 answer options (A–E). Random chance = 20%.

> Seed=42 · N=900 · Same question split as original paper run

In [1]:
# CELL 1 — Install

!pip install -q trl
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 14.3 MB/s eta 0:00:0000:01
Done.


In [2]:
# CELL 2 — Login
from huggingface_hub import login
login("")  # paste your HF token
print("Login done")

Login done


In [3]:
# CELL 3 — Imports
import os, json, re, random, time
import torch
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
from tqdm.notebook import tqdm

OUTPUT_DIR = "/content/csqa_ft3b_solo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# CELL 4 — Config
# eval_seed=42, eval_n=900 MUST match paper exactly.
CONFIG = {
    "model_name"        : "Qwen/Qwen2.5-3B-Instruct",
    "dataset_name"      : "tau/commonsense_qa",
    "eval_split"        : "validation",
    "train_split"       : "train",
    "eval_seed"         : 42,
    "eval_n"            : 900,
    "max_train_samples" : 8000,    # CSQA train ~9741 examples
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "learning_rate"     : 2e-4,
    "num_epochs"        : 3,
    "batch_size"        : 4,
    "grad_accum"        : 4,
    "max_seq_length"    : 320,
    "max_new_tokens"    : 16,     # just a letter — very short
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"        : 100,
}
print("Config ready. eval_seed=42, eval_n=900.")

Config ready. eval_seed=42, eval_n=900.


In [5]:
# CELL 5 — Load CommonsenseQA
print("Loading CommonsenseQA...")
raw_ds = load_dataset(CONFIG["dataset_name"])

def normalise_csqa(item):
    q = item["question"].strip()
    choices = list(zip(item["choices"]["label"], item["choices"]["text"]))
    choice_str = "\n".join(f"{lbl}. {txt}" for lbl, txt in choices)
    full_q = f"{q}\n\n{choice_str}"
    return {
        "question" : full_q,
        "answer"   : item["answerKey"].strip().upper(),
        "concept"  : item.get("question_concept",""),
    }

# Eval pool — validation split
eval_pool = [normalise_csqa(x) for x in raw_ds[CONFIG["eval_split"]]]
# Train pool — train split
train_pool = [normalise_csqa(x) for x in raw_ds[CONFIG["train_split"]]]

print(f"Validation pool : {len(eval_pool)}")
print(f"Train pool      : {len(train_pool)}")
ans_dist = Counter(x["answer"] for x in eval_pool)
print(f"Answer distribution (eval): {dict(sorted(ans_dist.items()))}")

Loading CommonsenseQA...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

Validation pool : 1221
Train pool      : 9741
Answer distribution (eval): {'A': 239, 'B': 255, 'C': 241, 'D': 251, 'E': 235}


In [6]:
# CELL 6 — Eval / train split
# Eval: sample N=900 from validation pool with seed=42 — matches paper
random.seed(CONFIG["eval_seed"])
if CONFIG["eval_n"] < len(eval_pool):
    eval_data = random.sample(eval_pool, CONFIG["eval_n"])
else:
    eval_data = eval_pool

eval_questions = set(x["question"] for x in eval_data)

# Train: use CSQA train split, capped at max_train_samples
# Do NOT use validation examples for training
train_candidates = [x for x in train_pool
                    if x["question"] not in eval_questions]
random.seed(0)
if len(train_candidates) > CONFIG["max_train_samples"]:
    train_data = random.sample(train_candidates, CONFIG["max_train_samples"])
else:
    train_data = train_candidates

overlap = eval_questions & set(x["question"] for x in train_data)
print(f"Eval  : {len(eval_data)} questions (seed=42, val split)")
print(f"Train : {len(train_data)} questions (train split)")
print(f"Overlap: {len(overlap)} (must be 0)")

# Answer distribution in eval
ed = Counter(x["answer"] for x in eval_data)
print(f"Eval answer dist: {dict(sorted(ed.items()))}")

Eval  : 900 questions (seed=42, val split)
Train : 8000 questions (train split)
Overlap: 0 (must be 0)
Eval answer dist: {'A': 170, 'B': 194, 'C': 166, 'D': 190, 'E': 180}


In [7]:
# CELL 7 — SFT prompt format
# Model is trained to output only the answer letter (A/B/C/D/E).
# No explanation, no chain-of-thought — pure direct answer.

SYSTEM_PROMPT = (
    "You are a precise multiple choice answering assistant.\n"
    "Read the question and all options carefully.\n"
    "Respond with only the letter of the correct answer: A, B, C, D, or E."
)

def format_sft(item, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Question:\n{item['question']}"},
        {"role": "assistant", "content": item["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

print("SFT format ready.")
print(f"Example Q (first 100 chars): {train_data[0]["question"][:100]}")
print(f"Example A: {train_data[0]["answer"]}")

SFT format ready.
Example Q (first 100 chars): What is a child likely hoping to achieve by going to play?

A. have fun
B. enjoyment
C. rush
D. bein
Example A: A


In [8]:
# CELL 8 — Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer ready.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.


In [9]:
# CELL 9 — Prepare HF Dataset
train_formatted = [format_sft(x, tokenizer) for x in train_data]
hf_train = Dataset.from_list(train_formatted)
print(f"Training examples: {len(hf_train)}")
print(f"Sample (first 250 chars):\n{hf_train[0]["text"][:250]}")

Training examples: 8000
Sample (first 250 chars):
<|im_start|>system
You are a precise multiple choice answering assistant.
Read the question and all options carefully.
Respond with only the letter of the correct answer: A, B, C, D, or E.<|im_end|>
<|im_start|>user
Question:
What is a child likely h


In [10]:
# CELL 10 — Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
)
base_model.config.use_cache = False
base_model.enable_input_require_grads()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM: 3.09GB


In [11]:
# CELL 11 — Attach LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [12]:
# CELL 12 — Fine-tune
# ~15-20 min on T4. ARC train set is small so 5 epochs.


training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=1,        # ← down from 4
    gradient_accumulation_steps=16,       # ← up from 4 (keeps effective batch=16)
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="adafactor",                    # ← saves ~2GB vs Adam
    logging_steps=100,
    save_strategy="epoch",
    report_to="none",
    gradient_checkpointing=True,          # ← enable here too
    dataloader_num_workers=0,
    seed=42,
)




import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


trainer = SFTTrainer(
    model=model, train_dataset=hf_train,
    processing_class=tokenizer, args=training_args,
)
t0 = time.time()
trainer.train()
print(f"Training done in {(time.time()-t0)/60:.1f} min.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
100,1.184557
200,0.781294
300,0.768644
400,0.765368
500,0.753143
600,0.640942
700,0.633883
800,0.628268
900,0.629136
1000,0.625826


Training done in 243.5 min.


In [13]:
# CELL 13 — Save model
ft_model_path = f"{OUTPUT_DIR}/ft_model"
trainer.save_model(ft_model_path)
tokenizer.save_pretrained(ft_model_path)
print(f"Saved to {ft_model_path}")

Saved to /content/csqa_ft3b_solo/ft_model


In [14]:
# CELL 14 — Answer extraction (MCQ letter)
def extract_mcq_answer(text):
    text = text.strip()
    # 1. Single letter on first line
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        m = re.match(r"^\**([A-E])\**[.):,]?$", lines[0], re.IGNORECASE)
        if m: return m.group(1).upper()
    # 2. 'answer is X'
    m = re.search(r"(?:answer\s+is|answer:|correct\s+answer)\s*\**([A-E])\**",
                  text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # 3. Bold letter
    m = re.search(r"\*\*([A-E])\*\*", text)
    if m: return m.group(1).upper()
    # 4. Parenthesised letter
    m = re.search(r"\(([A-E])\)", text)
    if m: return m.group(1).upper()
    # 5. Any standalone A-E
    m = re.search(r"\b([A-E])\b", text)
    if m: return m.group(1).upper()
    return ""

# Tests
_t = ["A","**B**","answer is C","The answer: D","(E)"]
_e = ["A","B","C","D","E"]
ok = all(extract_mcq_answer(t)==e for t,e in zip(_t,_e))
print("Extractor:", "PASSED" if ok else "FAIL")

Extractor: PASSED


In [15]:
# CELL 15 — Reload model for eval
del model, base_model, trainer
torch.cuda.empty_cache()

eval_base = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
).eval()
ft_model = PeftModel.from_pretrained(eval_base, ft_model_path).eval()
print("Fine-tuned model loaded for eval.")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Fine-tuned model loaded for eval.
VRAM: 6.37GB


In [16]:
# CELL 16 — Eval function (single greedy pass = 3.0B pp)
EVAL_SYSTEM = (
    "You are a precise multiple choice answering assistant.\n"
    "Read the question and all options carefully.\n"
    "Respond with only the letter of the correct answer: A, B, C, D, or E."
)

def run_ft_solo(question):
    messages = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": f"Question:\n{question}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=768)
    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,       # greedy — deterministic
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

# Verification: 20 questions
v_correct = 0; v_empty = 0
for item in eval_data[:20]:
    raw  = run_ft_solo(item["question"])
    pred = extract_mcq_answer(raw)
    if not pred: v_empty += 1
    if pred == item["answer"]: v_correct += 1
print(f"Verification (20 q): {v_correct}/20 = {v_correct/20*100:.0f}%")
print(f"Empty answers: {v_empty}/20")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Verification (20 q): 15/20 = 75%
Empty answers: 0/20


In [17]:
# CELL 17 — Full evaluation N=900
# ~15-20 min on T4 (single greedy pass per question)
print(f"Evaluating {CONFIG['eval_n']} questions | compute: 3.0B pp per question")
print("-"*60)

results = []; start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f: ck = json.load(f)
    start_idx = ck.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from {start_idx}")

t0 = time.time()
for idx in tqdm(range(start_idx, len(eval_data)), desc="CSQA-FT3B"):
    item = eval_data[idx]
    try:
        raw  = run_ft_solo(item["question"])
        pred = extract_mcq_answer(raw)
        gt   = item["answer"]
        results.append({
            "idx"          : idx,
            "question"     : item["question"],
            "gt_answer"    : gt,
            "raw_output"   : raw,
            "final_answer" : pred,
            "correct"      : (pred == gt),
            "empty"        : (pred == ""),
        })
    except Exception as e:
        results.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"], "raw_output": "",
            "final_answer": "", "correct": False,
            "empty": True, "error": str(e)
        })

    if (idx+1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in results: f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx+1}, f)
        acc  = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:4d}] acc={acc:.1f}%  ({mins:.1f}min)")

with open(CONFIG["results_file"], "w") as f:
    for r in results: f.write(json.dumps(r) + "\n")

n_correct = sum(r["correct"] for r in results)
n_empty   = sum(r["empty"]   for r in results)
print(f"\nAccuracy : {n_correct}/{len(results)} = {n_correct/len(results)*100:.1f}%")
print(f"Empty    : {n_empty}")

Evaluating 900 questions | compute: 3.0B pp per question
------------------------------------------------------------


CSQA-FT3B:   0%|          | 0/900 [00:00<?, ?it/s]

  [ 100] acc=83.0%  (0.3min)
  [ 200] acc=87.0%  (0.7min)
  [ 300] acc=84.7%  (1.0min)
  [ 400] acc=82.8%  (1.3min)
  [ 500] acc=82.4%  (1.7min)
  [ 600] acc=81.2%  (2.0min)
  [ 700] acc=81.1%  (2.3min)
  [ 800] acc=80.5%  (2.7min)
  [ 900] acc=80.6%  (3.0min)

Accuracy : 725/900 = 80.6%
Empty    : 0


In [18]:
# CELL 18 — Option distribution (position bias check)
# Compare FT 3B Solo letter dist against paper values
ft_dist = Counter(r["final_answer"] for r in results)
gt_dist  = Counter(r["gt_answer"]    for r in results)

print("Option distribution — FT 3B Solo vs Expected:")
print(f"  {"Opt":<5} {"FT Solo":>10} {"GT (label)":>12} {"Uniform":>10}")
print("-"*42)
for opt in ["A","B","C","D","E",""]:
    label = "(empty)" if opt == "" else opt
    ft_n  = ft_dist.get(opt, 0)
    gt_n  = gt_dist.get(opt, 0)
    uniform = len(results) / 5 if opt != "" else 0
    print(f"  {label:<5} {ft_n:>8} ({ft_n/len(results)*100:4.1f}%)  {gt_n:>8} ({gt_n/len(results)*100:4.1f}%)  {uniform:>8.0f} (20.0%)")

# Paper reference values
print("\nPaper reference:")
print("  Baseline E-bias: 24.8%  |  Guided E-rate: 14.0%  |  Expected: 20%")

Option distribution — FT 3B Solo vs Expected:
  Opt      FT Solo   GT (label)    Uniform
------------------------------------------
  A          176 (19.6%)       170 (18.9%)       180 (20.0%)
  B          193 (21.4%)       194 (21.6%)       180 (20.0%)
  C          172 (19.1%)       166 (18.4%)       180 (20.0%)
  D          178 (19.8%)       190 (21.1%)       180 (20.0%)
  E          181 (20.1%)       180 (20.0%)       180 (20.0%)
  (empty)        0 ( 0.0%)         0 ( 0.0%)         0 (20.0%)

Paper reference:
  Baseline E-bias: 24.8%  |  Guided E-rate: 14.0%  |  Expected: 20%


In [19]:
# CELL 19 — Final comparison table
ft_acc = sum(r["correct"] for r in results) / len(results) * 100

# Paper confirmed values for CSQA
BASELINE = 70.7; COT = 67.7; GUIDED = 75.8

print("="*65)
print("CommonsenseQA — FULL COMPUTE-ACCURACY COMPARISON")
print("="*65)
print(f"  Condition              | Compute   | Accuracy")
print(f"  -----------------------|-----------|----------")
print(f"  CoT (1.5B×5)           | 7.5B pp   | {COT}%  (CoT hurts — paper)")
print(f"  Baseline (1.5B×5)      | 7.5B pp   | {BASELINE}%  (paper)")
print(f"  FT 3B Solo (this run)  | 3.0B pp   | {ft_acc:.1f}%")
print(f"  Guided pipeline        | 10.5B pp  | {GUIDED}%  (paper)")
print()
gap_vs_guided  = GUIDED  - ft_acc
gap_vs_base    = ft_acc  - BASELINE
gap_vs_cot     = ft_acc  - COT
print(f"  FT Solo vs Baseline  : {gap_vs_base:+.1f} pts  (at 2.5x less compute)")
print(f"  FT Solo vs CoT       : {gap_vs_cot:+.1f} pts")
print(f"  Guided vs FT Solo    : {gap_vs_guided:+.1f} pts  (guided costs +7.5B pp more)")
print()
if ft_acc >= 74.0:
    print("  VERDICT: FT Solo matches guided. Architecture not justified on CSQA.")
elif ft_acc >= 70.0:
    print(f"  VERDICT: FT Solo matches baseline. Guided adds {gap_vs_guided:.1f} pts at 3.5x compute.")
elif ft_acc >= 60.0:
    print(f"  VERDICT: FT Solo below baseline. Guided pipeline adds clear value.")
else:
    print(f"  VERDICT: FT Solo underperforms. Fine-tuning on CSQA alone is insufficient.")

CommonsenseQA — FULL COMPUTE-ACCURACY COMPARISON
  Condition              | Compute   | Accuracy
  -----------------------|-----------|----------
  CoT (1.5B×5)           | 7.5B pp   | 67.7%  (CoT hurts — paper)
  Baseline (1.5B×5)      | 7.5B pp   | 70.7%  (paper)
  FT 3B Solo (this run)  | 3.0B pp   | 80.6%
  Guided pipeline        | 10.5B pp  | 75.8%  (paper)

  FT Solo vs Baseline  : +9.9 pts  (at 2.5x less compute)
  FT Solo vs CoT       : +12.9 pts
  Guided vs FT Solo    : -4.8 pts  (guided costs +7.5B pp more)

  VERDICT: FT Solo matches guided. Architecture not justified on CSQA.
